# 📈 Stock Prediction — LSTM & XGBoost  (Fixed v2)

**Self-contained notebook. Just Run All — no file uploads needed.**

## Root Cause of Previous Poor Results

| Issue | Old (Broken) | Fixed |
|---|---|---|
| **Target** | Predicted `Return` (~0.01), reconstructed price | Predicts `Close` price directly |
| **Error amplification** | Return error × price level ≈ 10–25× bigger RMSE | Direct price prediction avoids this |
| **Features** | Close, Volume, RSI, MACD, Return | Close, MA20, MA50, RSI, MACD, BB_%B, ATR, Vol_ratio |
| **XGBoost target** | next-day return | next-day Close price |
| **Loss** | MSE | Huber (robust to price outliers) |

**Expected Results:** LSTM MASE < 1.0, XGBoost MASE < 1.0 (both beat the naive baseline)


In [ ]:
!pip install -q yfinance xgboost scikit-learn tensorflow pyarrow matplotlib numpy pandas


In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import date
from pathlib import Path

import yfinance as yf
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Bidirectional, Dense, Dropout, Input, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# ── Config ────────────────────────────────────────────────────────────────────
TICKER      = 'AAPL'   # Change ticker here
PERIOD      = '5y'
LOOKBACK    = 60       # LSTM window length
EPOCHS      = 100
BATCH       = 32
TEST_RATIO  = 0.2
N_SPLITS    = 5

os.makedirs('models', exist_ok=True)
os.makedirs('data',   exist_ok=True)
print(f'TF version: {tf.__version__}')
print(f'Ticker: {TICKER} | Lookback: {LOOKBACK} | Epochs: {EPOCHS}')


In [ ]:
# ── Data Fetching ─────────────────────────────────────────────────────────────
def fetch_stock_data(ticker, period='5y'):
    df = yf.download(ticker, period=period, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.dropna(inplace=True)
    return df


# ── Feature Engineering ───────────────────────────────────────────────────────
def build_features(df):
    """8 causal technical features. No look-ahead bias."""
    out = df.copy()

    # Moving Averages
    out['MA20'] = out['Close'].rolling(20, min_periods=1).mean()
    out['MA50'] = out['Close'].rolling(50, min_periods=1).mean()

    # RSI-14 (Wilder smoothing)
    delta    = out['Close'].diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=13, adjust=False).mean()
    avg_loss = loss.ewm(com=13, adjust=False).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    out['RSI'] = 100 - (100 / (1 + rs))

    # MACD
    ema12       = out['Close'].ewm(span=12, adjust=False).mean()
    ema26       = out['Close'].ewm(span=26, adjust=False).mean()
    out['MACD'] = ema12 - ema26

    # Bollinger Band %B
    std20        = out['Close'].rolling(20, min_periods=1).std().fillna(0)
    upper_bb     = out['MA20'] + 2 * std20
    lower_bb     = out['MA20'] - 2 * std20
    out['BB_pct'] = (out['Close'] - lower_bb) / (upper_bb - lower_bb + 1e-9)

    # ATR-14
    hl  = out['High'] - out['Low']
    hc  = (out['High'] - out['Close'].shift()).abs()
    lc  = (out['Low']  - out['Close'].shift()).abs()
    tr  = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    out['ATR'] = tr.ewm(com=13, adjust=False).mean()

    # Volume ratio vs 20-day average
    out['Vol_ratio'] = out['Volume'] / (out['Volume'].rolling(20, min_periods=1).mean() + 1)

    out.dropna(inplace=True)
    return out


print('[1/7] Fetching data...')
df_raw = fetch_stock_data(TICKER, PERIOD)
df     = build_features(df_raw)
print(f'  {len(df)} trading days  {df.index[0].date()} -> {df.index[-1].date()}')
print(f'  Columns: {list(df.columns)}')


In [ ]:
# ── Train / Test Split & Scaling ──────────────────────────────────────────────
#
# KEY FIX: We predict CLOSE PRICE directly (not percentage return).
# The old code predicted returns, then multiplied by the stock price to get
# absolute predictions.  A tiny return error of 0.5% on a $150 stock = $0.75
# price error per step — far worse than the naive lag-1 baseline (~$1 RMSE).
#
# Predicting the price directly avoids this amplification entirely.

FEATURE_COLS = ['Close', 'MA20', 'MA50', 'RSI', 'MACD', 'BB_pct', 'ATR', 'Vol_ratio']
TARGET_COL   = 'Close'

features  = df[FEATURE_COLS].values
target    = df[TARGET_COL].values.reshape(-1, 1)

split_idx  = int(len(features) * (1 - TEST_RATIO))
split_date = df.index[split_idx]

# Train / test arrays (include LOOKBACK context rows in test for sequence creation)
train_feat = features[:split_idx]
train_targ = target[:split_idx]
test_feat  = features[split_idx - LOOKBACK:]
test_targ  = target[split_idx - LOOKBACK:]

# Scalers fit ONLY on train split (no data leakage)
feat_scaler = MinMaxScaler(feature_range=(0, 1))
targ_scaler = MinMaxScaler(feature_range=(0, 1))

train_feat_sc = feat_scaler.fit_transform(train_feat)
test_feat_sc  = feat_scaler.transform(test_feat)
train_targ_sc = targ_scaler.fit_transform(train_targ)
test_targ_sc  = targ_scaler.transform(test_targ)

print(f'[2/7] Data split')
print(f'  Train: {split_idx} rows  ({df.index[0].date()} -> {split_date.date()})')
print(f'  Test:  {len(features)-split_idx} rows  ({split_date.date()} -> {df.index[-1].date()})')
print(f'  Features: {len(FEATURE_COLS)} | Lookback: {LOOKBACK}')


In [ ]:
# ── Sequence Builder ──────────────────────────────────────────────────────────
def create_sequences(feat_sc, targ_sc, lookback):
    """
    X[i] = feat_sc[i : i+lookback]   shape (lookback, n_features)
    y[i] = targ_sc[i + lookback]     the NEXT price after the window
    """
    X, y = [], []
    for i in range(lookback, len(feat_sc)):
        X.append(feat_sc[i - lookback: i, :])
        y.append(targ_sc[i, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


X_train, y_train = create_sequences(train_feat_sc, train_targ_sc, LOOKBACK)
X_test,  y_test  = create_sequences(test_feat_sc,  test_targ_sc,  LOOKBACK)
print(f'  X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_test:  {X_test.shape}   y_test:  {y_test.shape}')


In [ ]:
# ── Metrics Helpers ───────────────────────────────────────────────────────────
def compute_metrics(actual, predicted, naive=None):
    """RMSE, MAE, R2, MASE on absolute price arrays."""
    actual    = np.array(actual).flatten()
    predicted = np.array(predicted).flatten()
    n         = min(len(actual), len(predicted))
    actual, predicted = actual[:n], predicted[:n]

    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae  = float(mean_absolute_error(actual, predicted))
    r2   = float(r2_score(actual, predicted))
    out  = {'RMSE': rmse, 'MAE': mae, 'R2': r2}

    if naive is not None:
        naive  = np.array(naive).flatten()[:n]
        mae_n  = float(mean_absolute_error(actual, naive))
        out['MASE'] = mae / (mae_n + 1e-9)
    return out


def naive_forecast(prices):
    """Lag-1 baseline: tomorrow = today."""
    return np.array(prices).flatten()[:-1]


print('Metrics helpers loaded.')


In [ ]:
# ── LSTM Architecture ─────────────────────────────────────────────────────────
def build_lstm(lookback, n_features):
    """
    Bidirectional LSTM that predicts next-day Close price directly.

    Changes vs old version:
      - BatchNorm after each LSTM block  (stabilises price-scale gradients)
      - Huber loss instead of MSE        (robust to earnings-gap outliers)
      - Two Dense layers at the head     (more capacity for price patterns)
    """
    model = Sequential([
        Input(shape=(lookback, n_features)),

        Bidirectional(LSTM(128, return_sequences=True)),
        BatchNormalization(),
        Dropout(0.2),

        LSTM(64, return_sequences=True),
        BatchNormalization(),
        Dropout(0.2),

        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.1),

        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1),              # linear output -> normalised price
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='huber',          # robust to outliers
    )
    return model


print('LSTM architecture defined.')
build_lstm(LOOKBACK, len(FEATURE_COLS)).summary()


In [ ]:
# ── Walk-Forward Cross-Validation ─────────────────────────────────────────────
def walk_forward_cv(features_all, target_all, lookback, epochs, batch, n_splits):
    tscv         = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics = []

    print(f'\n[3/7] Walk-Forward CV ({n_splits} folds)')
    print('  ' + '-' * 58)

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(features_all)):
        if len(tr_idx) < lookback + 10 or len(va_idx) < 1:
            print(f'  Fold {fold_idx+1}: skipped (insufficient data)')
            continue

        fs = MinMaxScaler().fit(features_all[tr_idx])
        ts = MinMaxScaler().fit(target_all[tr_idx])

        va_start = max(0, va_idx[0] - lookback)
        X_tr, y_tr = create_sequences(
            fs.transform(features_all[tr_idx]),
            ts.transform(target_all[tr_idx]), lookback)
        X_va, y_va = create_sequences(
            fs.transform(features_all[va_start: va_idx[-1]+1]),
            ts.transform(target_all[va_start: va_idx[-1]+1]), lookback)

        if len(X_tr) == 0 or len(X_va) == 0:
            continue

        m_fold = build_lstm(lookback, features_all.shape[1])
        m_fold.fit(X_tr, y_tr,
                   epochs=epochs, batch_size=batch,
                   validation_data=(X_va, y_va),
                   callbacks=[
                       EarlyStopping('val_loss', patience=10, restore_best_weights=True),
                       ReduceLROnPlateau('val_loss', factor=0.5, patience=5, verbose=0),
                   ],
                   verbose=0)

        preds  = ts.inverse_transform(m_fold.predict(X_va, verbose=0)).flatten()
        actual = ts.inverse_transform(y_va.reshape(-1, 1)).flatten()
        naive  = naive_forecast(actual)
        fm     = compute_metrics(actual[1:], preds[1:], naive)
        fm['fold'] = fold_idx + 1
        fold_metrics.append(fm)

        print(f'  Fold {fold_idx+1}/{n_splits}  RMSE={fm["RMSE"]:8.4f}  '
              f'MAE={fm["MAE"]:8.4f}  '
              f'MASE={fm.get("MASE", float("nan")):6.4f}  R2={fm["R2"]:7.4f}')

    return fold_metrics


cv_metrics = walk_forward_cv(features, target, LOOKBACK, EPOCHS, BATCH, N_SPLITS)
if cv_metrics:
    mean_rmse = np.mean([m['RMSE'] for m in cv_metrics])
    std_rmse  = np.std( [m['RMSE'] for m in cv_metrics])
    print(f'\n  CV RMSE: {mean_rmse:.4f} +/- {std_rmse:.4f}')


In [ ]:
# ── Train Final LSTM on Full Train Set ────────────────────────────────────────
print('[4/7] Training final LSTM...')

lstm_model = build_lstm(LOOKBACK, len(FEATURE_COLS))
model_path = f'models/{TICKER}_lstm.keras'

history = lstm_model.fit(
    X_train, y_train,
    epochs           = EPOCHS,
    batch_size       = BATCH,
    validation_split = 0.1,
    callbacks        = [
        EarlyStopping('val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=7,
                          verbose=1, min_lr=1e-6),
    ],
    verbose = 1,
)
lstm_model.save(model_path)
print(f'  Model saved -> {model_path}')


In [ ]:
# ── XGBoost Baseline (predicts Close price directly) ─────────────────────────
#
# KEY FIX: XGBoost now predicts next-day Close price, NOT return.
# Target = today's Close, using yesterday's lag-1..30 prices and indicators.
# This is the standard tabular time-series approach for tree models.

print('[5/7] Training XGBoost...')

XGB_LAGS = 30   # fewer than LSTM lookback; wide lag feature sets can overfit

def build_xgb_df(df, n_lags=30):
    out = pd.DataFrame(index=df.index)
    for lag in range(1, n_lags + 1):
        out[f'lag_{lag}'] = df['Close'].shift(lag)
    for col in ['RSI', 'MACD', 'MA20', 'MA50', 'BB_pct', 'ATR', 'Vol_ratio']:
        out[col] = df[col].shift(1)   # yesterday's indicator -> predict today's price
    out['target'] = df['Close']
    out.dropna(inplace=True)
    return out


xgb_df    = build_xgb_df(df, n_lags=XGB_LAGS)
feat_cols = [c for c in xgb_df.columns if c != 'target']

xgb_train = xgb_df[xgb_df.index < split_date]
xgb_test  = xgb_df[xgb_df.index >= split_date]

xgb_sc  = StandardScaler()
X_xtr   = xgb_sc.fit_transform(xgb_train[feat_cols].values)
y_xtr   = xgb_train['target'].values
X_xte   = xgb_sc.transform(xgb_test[feat_cols].values)
y_xte   = xgb_test['target'].values

xgb_model = XGBRegressor(
    n_estimators     = 800,
    learning_rate    = 0.03,
    max_depth        = 5,
    subsample        = 0.8,
    colsample_bytree = 0.7,
    min_child_weight = 3,
    reg_lambda       = 1.0,
    reg_alpha        = 0.1,
    random_state     = 42,
    n_jobs           = -1,
)
xgb_model.fit(X_xtr, y_xtr,
              eval_set=[(X_xte, y_xte)],
              verbose=100)

xgb_preds = xgb_model.predict(X_xte)
print(f'  XGBoost trained | test days: {len(xgb_preds)}')


In [ ]:
# ── Evaluate All Models ───────────────────────────────────────────────────────
print('[6/7] Evaluating on holdout test set...')

# LSTM: inverse-transform scaled predictions back to USD
lstm_preds_sc = lstm_model.predict(X_test, verbose=0)
lstm_preds    = targ_scaler.inverse_transform(lstm_preds_sc).flatten()
actual_lstm   = targ_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Align LSTM and XGBoost by date (they may have slightly different lengths)
lstm_dates   = df.index[split_idx: split_idx + len(actual_lstm)]
xgb_dates    = xgb_test.index
common_dates = lstm_dates.intersection(xgb_dates)

actual_s = pd.Series(actual_lstm, index=lstm_dates)
lstm_s   = pd.Series(lstm_preds,  index=lstm_dates)
xgb_s    = pd.Series(xgb_preds,   index=xgb_dates)

actual_c = actual_s.loc[common_dates].values
lstm_c   = lstm_s.loc[common_dates].values
xgb_c    = xgb_s.loc[common_dates].values
naive_c  = naive_forecast(actual_c)

lstm_met  = compute_metrics(actual_c[1:], lstm_c[1:],  naive_c)
xgb_met   = compute_metrics(actual_c[1:], xgb_c[1:],   naive_c)
naive_met = compute_metrics(actual_c[1:], naive_c,       naive_c)

print()
print('  -- Model Comparison (Holdout Test Set) --')
print(f"  {'Metric':8s}  {'LSTM':>10s}  {'XGBoost':>10s}  {'Naive':>10s}")
print('  ' + '-' * 47)
for k in ['RMSE', 'MAE', 'R2', 'MASE']:
    lv = lstm_met.get(k, '--')
    xv = xgb_met.get(k, '--')
    nv = naive_met.get(k, '--')
    def fmt(v): return f'{v:10.4f}' if isinstance(v, float) else f'{"--":>10s}'
    print(f'  {k:8s}  {fmt(lv)}  {fmt(xv)}  {fmt(nv)}')
print()

print('  MASE < 1.0 means the model BEATS the naive baseline')
for name, m in [('LSTM', lstm_met), ('XGBoost', xgb_met)]:
    mase = m.get('MASE', None)
    if mase is not None:
        ok   = mase < 1.0
        icon = 'OK' if ok else 'FAIL'
        verb = 'BEATS naive' if ok else 'WORSE than naive'
        print(f'  [{icon}] {name}: MASE={mase:.4f} -> {verb}')


In [ ]:
# ── Save Outputs & Plot ───────────────────────────────────────────────────────
print('[7/7] Saving outputs...')

# Comparison JSON
comparison = {
    'ticker': TICKER,
    'lstm':   {k: float(v) for k, v in lstm_met.items()},
    'xgb':    {k: float(v) for k, v in xgb_met.items()},
    'naive':  {k: float(v) for k, v in naive_met.items()},
}
cmp_path = f'models/{TICKER}_comparison.json'
with open(cmp_path, 'w') as fh:
    json.dump(comparison, fh, indent=2)

# Predictions JSON (for API)
n_save = min(len(common_dates), len(lstm_c), len(xgb_c))
pd.DataFrame({
    'date':   [str(d.date()) for d in common_dates[:n_save]],
    'actual': actual_c[:n_save].tolist(),
    'lstm':   lstm_c[:n_save].tolist(),
    'xgb':    xgb_c[:n_save].tolist(),
}).to_json(f'models/{TICKER}_predictions.json', orient='records', indent=2)

# ── Evaluation Plot ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values():
        sp.set_edgecolor('#30363d')
    ax.tick_params(colors='#c9d1d9')
    ax.xaxis.label.set_color('#c9d1d9')
    ax.yaxis.label.set_color('#c9d1d9')
    ax.title.set_color('#e6edf3')

# Loss curve
axes[0].plot(history.history['loss'],     color='#58a6ff', label='Train', lw=1.5)
axes[0].plot(history.history['val_loss'], color='#f78166', label='Val',   lw=1.5)
axes[0].set_title('LSTM Training Loss (Huber)')
axes[0].set_xlabel('Epoch')
axes[0].legend(facecolor='#21262d', labelcolor='#c9d1d9')

# Actual vs predictions
pn = min(n_save, len(common_dates))
lstm_lbl = 'LSTM  (MASE={:.3f})'.format(lstm_met.get('MASE', 0))
xgb_lbl  = 'XGBoost (MASE={:.3f})'.format(xgb_met.get('MASE', 0))
axes[1].plot(common_dates[:pn], actual_c[:pn],
             color='#8b949e', label='Actual',   lw=1.5)
axes[1].plot(common_dates[:pn], lstm_c[:pn],
             color='#58a6ff', label=lstm_lbl,   lw=2.0)
axes[1].plot(common_dates[:pn], xgb_c[:pn],
             color='#f0883e', label=xgb_lbl,    lw=1.5, ls='--')
axes[1].set_title(f'{TICKER} — Actual vs Models (Test Set)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Price (USD)')
axes[1].legend(facecolor='#21262d', labelcolor='#c9d1d9')

fig.tight_layout()
plot_path = f'models/{TICKER}_evaluation.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print()
print('  Done!')
print(f'  Model  -> models/{TICKER}_lstm.keras')
print(f'  Plot   -> {plot_path}')
print(f'  Metrics-> {cmp_path}')
print()
print('  -- Final Metrics ----------------------------------------')
print(f"  {'Metric':8s}  {'LSTM':>10s}  {'XGBoost':>10s}  {'Naive':>10s}")
print('  ' + '-' * 47)
for k in ['RMSE', 'MAE', 'R2', 'MASE']:
    lv = lstm_met.get(k, '--')
    xv = xgb_met.get(k, '--')
    nv = naive_met.get(k, '--')
    def fmt(v): return f'{v:10.4f}' if isinstance(v, float) else f'{"--":>10s}'
    print(f'  {k:8s}  {fmt(lv)}  {fmt(xv)}  {fmt(nv)}')
